# Parametric Analysis with `MorphingIterator`

This notebook demonstrates the most powerful feature of the `pyfwg` library: the `MorphingIterator` class.

This class is designed for running **parametric studies**, where you need to execute the morphing process multiple times with different parameters (e.g., different GCMs, interpolation methods, or even different EPW files). It uses a Pandas DataFrame to define all the runs in a structured and easy-to-manage way.

**Supported binaries:** Global 5.2.0, Europe EUR-11 3.2.0 and the other eight CORDEX-CMIP5 distributions 1.2.0. Older Global/Europe releases are outside the supported scope.

**New in pyfwg v0.4.0:** `pyfwg` also supports the **FWG Engine** distributions (**Global v5.x**, **Europe v3.x** and **CORDEX regional**). The last section of this notebook runs a parametric study with Global v5.x that compares two uncertainty cases of the climate-change signal.


## Configuration

The supported examples use Global 5.2.0, Europe EUR-11 3.2.0 and the other eight CORDEX-CMIP5 distributions 1.2.0. Older Global/Europe releases are outside the supported scope. Download JARs from the [FWG website](https://future-weather-generator.adai.pt/).

Paths are read from environment variables first, then an optional `fwg.local.json` in the project/working folder, then the defaults below. This local JSON file is excluded from Git. JARs can be on an external drive; keep it connected throughout a run. A path containing spaces needs no special escaping in a Python string.

- `FWG_JAR_DIR`: folder of regional JARs (default: project root); keep one version per domain.
- `FWG_GLOBAL_JAR_PATH`: optional full path to Global 5.2.0 if it is in another folder.
- `FWG_EPW_DIR`: real regional baselines named `<domain>.epw`. Global examples use `EUR-11.epw`. Retained validation samples are used if present; otherwise supply `epws/cordex`. Public sample links are in the regional notebook. Do not alter station coordinates to fit a region.
- `FWG_CACHE_DIR`: extracted climate data (default: `~/fwg_model_cache`). EUR-11's full ensemble alone contains about 11.3 GB before extraction overhead. Cache location is independent of the JAR folder.
- `FWG_OUTPUT_DIR`: results, Excel workbooks and diagnostics (default: `build/cordex-tutorial`).
- `FWG_JAVA`: optional Java executable; otherwise Java 17+ must be on PATH.

`RUN_MORPHING = False` prepares every example without starting Java. Set it to `True` to generate weather. Notebook outputs are cleared before distribution; generated files are kept for inspection.


In [ ]:
import json
import os
from pathlib import Path
from pyfwg import enable_console_logging

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyfwg" / "__init__.py").is_file()),
    Path.cwd(),
)
local_file = PROJECT_ROOT / "fwg.local.json"
local_paths = json.loads(local_file.read_text(encoding="utf-8")) if local_file.is_file() else {}

def configured_path(name, default):
    return Path(os.environ.get(name, local_paths.get(name, str(default)))).expanduser().resolve()

validation_epws = PROJECT_ROOT / "build/cordex-validation/epws"
default_epws = validation_epws if validation_epws.is_dir() else PROJECT_ROOT / "epws/cordex"
FWG_JAR_DIR = configured_path("FWG_JAR_DIR", PROJECT_ROOT)
FWG_GLOBAL_JAR_PATH = configured_path("FWG_GLOBAL_JAR_PATH", FWG_JAR_DIR / "FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar")
FWG_EPW_DIR = configured_path("FWG_EPW_DIR", default_epws)
FWG_CACHE_DIR = configured_path("FWG_CACHE_DIR", Path.home() / "fwg_model_cache")
FWG_OUTPUT_DIR = configured_path("FWG_OUTPUT_DIR", PROJECT_ROOT / "build/cordex-tutorial")
java_executable = os.environ.get("FWG_JAVA", local_paths.get("FWG_JAVA"))
if java_executable:
    java_path = Path(java_executable).expanduser().resolve()
    if not java_path.is_file():
        raise FileNotFoundError(f"Java executable not found: {java_path}")
    os.environ["PATH"] = str(java_path.parent) + os.pathsep + os.environ.get("PATH", "")
RUN_MORPHING = False
enable_console_logging()
print("Regional JARs:", FWG_JAR_DIR)


### Read a regional JAR and select compatible inputs

This helper reads the actual manifest, including renamed JARs. It selects one model by default, or every ensemble-eligible model with `ensemble=True`, then finds a scenario and period shared by the selection. Empty selections and ambiguous JAR folders fail with an explicit message. The returned dictionary uses the public `pyfwg` parameters; inspect or edit it before running an example.

Caches are separated by distribution ID **and version**, since the same GCM–RCM name can occur in different regions. All examples use `IDW`. `BI`/`1` needs a rectilinear geographic grid and is rejected for these rotated/projected regional grids. One distinct model disables ensemble mode; aliases containing underscores and hyphens refer to the same model. CORDEX Engine distributions do not implement UHI.


In [ ]:
from pyfwg import get_fwg_distribution_info

def regional_jar(domain):
    matches = []
    for jar in sorted(FWG_JAR_DIR.glob("*.jar")):
        info = get_fwg_distribution_info(str(jar))
        if info and info["framework"] == "CORDEX" and info["domain"] == domain:
            matches.append((jar, info))
    if len(matches) != 1:
        raise ValueError(f"Expected one {domain} JAR in {FWG_JAR_DIR}; found {len(matches)}. Check the folder and versions.")
    return matches[0]

def regional_inputs(domain, ensemble=False):
    jar, info = regional_jar(domain)
    models = info["ensemble_models"] if ensemble else info["models"][:1]
    if not models or (ensemble and len(models) < 2):
        raise ValueError(f"{domain}: need one model, or at least two eligible models for an ensemble.")
    details = info["model_details"]
    scenarios = [s for s in info["scenarios"] if all(s in details[m]["scenarios"] for m in models)]
    periods = [t for t in info["timeframes"] if all(t in details[m]["timeframes"] for m in models)]
    if not scenarios or not periods:
        raise ValueError(f"{domain}: selected models have no shared scenario or period.")
    epw = FWG_EPW_DIR / f"{domain}.epw"
    if not epw.is_file():
        raise FileNotFoundError(f"Supply a real baseline inside {domain} at {epw}, or set FWG_EPW_DIR.")
    options = dict(
        epw_paths=str(epw), fwg_jar_path=str(jar),
        fwg_rcm_pairs=list(models), fwg_create_ensemble=ensemble,
        fwg_scenarios=[scenarios[0]], fwg_timeframes=[periods[0]],
        fwg_interpolation_method_id="IDW", fwg_output_profile="MINIMAL",
        fwg_model_unzipped_folder=str(FWG_CACHE_DIR / info["distribution_id"] / info["version"]),
        fwg_extra_args={"threads": 1, "io_threads": 1, "save_technical_log": True},
        delete_temp_files=False,
    )
    return info, options


In [ ]:
def global_inputs():
    info = get_fwg_distribution_info(str(FWG_GLOBAL_JAR_PATH))
    if not info or info["framework"] != "CMIP" or info.get("domain"):
        raise ValueError(f"Supply a Global FWG Engine JAR at {FWG_GLOBAL_JAR_PATH}")
    epw = FWG_EPW_DIR / "EUR-11.epw"
    if not epw.is_file():
        raise FileNotFoundError(f"Supply the Global example baseline at {epw}")
    model = "CanESM5" if "CanESM5" in info["models"] else info["models"][0]
    details = info["model_details"][model]
    scenario = next(s for s in info["scenarios"] if s in details["scenarios"])
    period = next(t for t in info["timeframes"] if t in details["timeframes"])
    return info, dict(
        epw_paths=str(epw), fwg_jar_path=str(FWG_GLOBAL_JAR_PATH),
        fwg_gcms=[model], fwg_create_ensemble=False, fwg_add_uhi=False,
        fwg_scenarios=[scenario], fwg_timeframes=[int(period)],
        fwg_interpolation_method_id="IDW", fwg_output_profile="MINIMAL",
        fwg_use_multithreading=False,
        fwg_model_unzipped_folder=str(FWG_CACHE_DIR / info["distribution_id"] / info["version"]),
        fwg_extra_args={"threads": 1, "io_threads": 1, "save_technical_log": True},
        delete_temp_files=False,
    )


## Global 5.2.0: prepare a DataFrame batch

Create the iterator with the class appropriate to the JAR. Each row describes a run; a row's explicit values override iterator defaults. Empty cells retain defaults. The example compares two month-transition settings while keeping the same model, SSP and period. The filename includes the varying parameter to avoid collisions.


In [ ]:
import pandas as pd
from pyfwg import MorphingIterator, MorphingWorkflowGlobal
info_global, global_options = global_inputs()
iterator_global = MorphingIterator(MorphingWorkflowGlobal)
iterator_global.set_default_values(
    **{k: v for k, v in global_options.items() if k != "epw_paths"},
    output_filename_pattern="{station}_{ssp}_{year}_smooth-{fwg_month_transition_hours}",
)
runs_global = pd.DataFrame([
    dict(epw_paths=[global_options["epw_paths"]],
         final_output_dir=str(FWG_OUTPUT_DIR / "global-iterator"),
         temp_base_dir=str(FWG_OUTPUT_DIR / "temp-global-iterator" / str(hours)),
         keyword_mapping={"station": {"Seville": "EUR-11"}},
         fwg_month_transition_hours=hours)
    for hours in (24, 48)
])
iterator_global.generate_morphing_workflows(runs_global)
if len(iterator_global.prepared_workflows) != len(runs_global) or not all(
    w.is_config_valid for w in iterator_global.prepared_workflows
):
    raise ValueError("Review the Global iterator configuration warnings.")
display(iterator_global.morphing_workflows_plan_df)
if RUN_MORPHING:
    iterator_global.run_morphing_workflows(show_tool_output=False)


A Global iterator cannot run a regional JAR. Use `MorphingIterator(MorphingWorkflowEurope)` for EUR-11, or `MorphingIterator(MorphingWorkflowCordex)` for a batch spanning CORDEX domains. The same Excel export/load functions work with all three classes. The next example demonstrates an actual populated workbook.


## A batch spanning CORDEX regions

Use `MorphingIterator(MorphingWorkflowCordex)` and put a different JAR, baseline, cache and scenario selection in each DataFrame row. Keep common defaults independent of any one region. For example, a shared `rcp45` default would be invalid for MNA-22.

Start with NAM-44 (Toronto) and MNA-22 (Cairo). Change `STUDY_DOMAINS` to any subset of the nine domains, or `list(CORDEX_REGIONS)` if every JAR and baseline is available. The helper selects one model per region; change `ensemble=False` to `True` for full ensembles. Outputs include `{region}` and also use separate directories, preventing collisions between regions that share scenario/year labels.

The iterator runs sequentially and retains each shared climate cache. It does not remove the cache of a previous region. With limited disk space, use one domain per batch and remove its specific cache after reviewing the results. Set `RUN_MORPHING=True` only when ready to generate weather.


In [ ]:
import pandas as pd
from pyfwg import MorphingIterator, MorphingWorkflowCordex, CORDEX_REGIONS

STUDY_DOMAINS = ["NAM-44", "MNA-22"]
regional_rows = []
for domain in STUDY_DOMAINS:
    info, options = regional_inputs(domain, ensemble=False)
    regional_rows.append({
        **options,
        "final_output_dir": str(FWG_OUTPUT_DIR / "iterator" / domain),
        "temp_base_dir": str(FWG_OUTPUT_DIR / "temp-iterator" / domain),
    })
runs_df_cordex = pd.DataFrame(regional_rows)
iterator_cordex = MorphingIterator(MorphingWorkflowCordex)
iterator_cordex.set_default_values(output_filename_pattern="{region}_{rcp}_{year}")
iterator_cordex.generate_morphing_workflows(
    runs_df=runs_df_cordex,
    keyword_mapping={"region": {domain: domain for domain in STUDY_DOMAINS}},
)
if not all(workflow.is_config_valid for workflow in iterator_cordex.prepared_workflows):
    raise ValueError("Review the regional configuration warnings before running the batch.")
runs_df_cordex[["epw_paths", "fwg_scenarios", "fwg_timeframes", "final_output_dir"]]


In [ ]:
if RUN_MORPHING:
    iterator_cordex.run_morphing_workflows(show_tool_output=False)
else:
    print("Batch prepared; set RUN_MORPHING=True to generate weather.")


## Run the regional batch from an Excel workbook

This route exports a blank template and creates a populated example from the regional DataFrame above. It then reloads the XLSX through `load_runs_from_excel`, prepares the iterator, and runs it only when `RUN_MORPHING=True`. Existing workbooks are kept so that your edits are not overwritten. Edit `cordex-runs.xlsx`, save it, and rerun the loading cell to apply your changes.

List and dictionary cells use Python notation, for example `['rcp85']`, `[2050]` and `{'threads': 1}`. Leave a cell empty to use the iterator/workflow default. Explicit parameter cells override values in `fwg_params`. Choose a scenario and model list supported by each row's JAR. The output and temporary folders in this workbook are separate from the preceding DataFrame example.


In [ ]:
from pyfwg import export_template_to_excel, load_runs_from_excel

iterator_xlsx = MorphingIterator(MorphingWorkflowCordex)
iterator_xlsx.set_default_values(output_filename_pattern="{region}_{rcp}_{year}")
FWG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
template_path = FWG_OUTPUT_DIR / "cordex-template.xlsx"
runs_path = FWG_OUTPUT_DIR / "cordex-runs.xlsx"
if not template_path.exists():
    export_template_to_excel(iterator_xlsx, str(template_path))
if not runs_path.exists():
    excel_rows = []
    for row in runs_df_cordex.to_dict("records"):
        domain = Path(row["epw_paths"]).stem
        excel_rows.append({
            **row,
            "epw_paths": [row["epw_paths"]],
            "keyword_mapping": {"region": {domain: domain}},
            "final_output_dir": str(FWG_OUTPUT_DIR / "iterator-xlsx" / domain),
            "temp_base_dir": str(FWG_OUTPUT_DIR / "temp-iterator-xlsx" / domain),
        })
    # Keep every exported template column, including empty optional settings.
    pd.DataFrame(excel_rows).reindex(columns=iterator_xlsx.get_template_dataframe().columns).to_excel(runs_path, index=False)
print("Blank template:", template_path)
print("Edit this populated workbook:", runs_path)


In [ ]:
runs_from_excel = load_runs_from_excel(str(runs_path))
iterator_xlsx.generate_morphing_workflows(runs_from_excel)
if len(iterator_xlsx.prepared_workflows) != len(runs_from_excel) or not all(
    workflow.is_config_valid for workflow in iterator_xlsx.prepared_workflows
):
    raise ValueError("Some Excel rows did not produce valid workflows; inspect the configuration warnings.")
runs_from_excel[["epw_paths", "fwg_rcm_pairs", "fwg_scenarios", "fwg_timeframes", "final_output_dir"]]


In [ ]:
if RUN_MORPHING:
    iterator_xlsx.run_morphing_workflows(show_tool_output=False)
else:
    print("Excel batch prepared; set RUN_MORPHING=True to generate weather.")


### Inspect results, SolarQC and disk usage

For EPW output, each selected scenario/period normally produces a future EPW and its STAT. The API returns their paths; workflows move and rename them into `final_output_dir`. Check that files exist: a failed Java run is logged and may leave an empty result list. A preview alone does not establish that the EPW passes the engine's checks.

These examples keep `delete_temp_files=False` and technical logs. For SolarQC rejection, inspect `<temp_base_dir>/<EPW stem>/metadata/solarqc_preprocessing_failure.txt` and the `logs/application` folder. During the 2026-10-28 integration checks, New Delhi and Suvarnabhumi samples were rejected; unmodified public Mumbai and Singapore samples passed. Changing the JAR or station coordinates is not a repair for an unsuitable baseline. `MINIMAL` reduces auxiliary output; mandatory diagnostics can still be written.

After reviewing results and diagnostics, you can remove the corresponding temporary directory and the **specific distribution/version cache** when no run is using it. `delete_temp_files=True` removes per-EPW temporary files after a successful run; it does not clear a separately configured shared cache. No cleanup cell in these regional examples removes your results or caches automatically. An iterator reuses caches but does not evict previous regions: with limited space, run one region at a time and clean its cache before continuing.

The downloaded TMYx test samples cover 2009–2023, whereas the CORDEX climate reference period is 1976–2005. FWG reports this mismatch. These are software integration examples; choose and assess the baseline/reference period for your study. See the [CORDEX regional notebook](cordex_regions_example.ipynb) for the nine-region validation matrix and source links.
